# 1.5 为什么神经网络能够被训练：高维空间中的优化直觉

jshn9515  
2026-08-19

<a href="https://colab.research.google.com/github/jshn9515/dnnl-notebooks/blob/main/zh/ch1-introduction/ch1.5-why-neural-networks-can-be-trained.ipynb" data-fig-align="left"><img src="https://colab.research.google.com/assets/colab-badge.svg" /></a>

在前面的章节中，我们已经把神经网络的训练过程拆开来看了一遍。模型首先通过前向传播得到预测，再用损失函数衡量预测和目标之间的差距；反向传播负责计算梯度，而梯度下降则根据这些梯度更新参数。于是，一次最基本的训练可以写成：

$$
\text{Forward}
\rightarrow
\text{Loss}
\rightarrow
\text{Backward}
\rightarrow
\text{Update}
$$

从流程上看，这件事似乎已经解释完了。但如果我们再想深一点，会发现一个更奇怪的问题：

> **为什么这一套方法真的能够训练出神经网络？**

一个现代神经网络可能有几百万、几十亿甚至更多参数。它对应的损失函数 $L(\theta)$ 定义在一个极高维的参数空间中，而且通常是一个复杂的非凸函数。按照我们在二维图像中形成的直觉，这样的函数应该充满山峰、山谷和局部极小值。梯度下降每次只看当前位置附近的梯度，既看不到整个损失函数，也不知道全局最优点在哪里。那它为什么没有走几步就掉进某个小坑里，再也出不来了？

更进一步，神经网络的参数量往往大到足以记住训练数据。既然模型有这么多自由度，为什么它最后不仅能把训练损失降下来，还经常能够在没有见过的数据上工作？

这些问题到今天也没有一个简单、统一的答案。深度学习理论仍然在研究优化、表示学习和泛化之间到底是什么关系。不过，我们已经知道一些非常重要的直觉。它们能帮助我们理解：神经网络虽然是一个巨大的非凸优化问题，但它并没有二维图像看起来那么绝望。

## 1.5.1 Loss Landscape 并不是一张二维山谷图

我们经常把梯度下降画成一个小球沿着曲线向下滚：

<figure>
<img src="figures/ch1.5-loss-landscape.png" alt="图 1.5.1.1 梯度下降的二维直觉" width="70%" />
<figcaption aria-hidden="true">图 1.5.1.1 梯度下降的二维直觉</figcaption>
</figure>

这张图很有用，但也很容易带来一个错误直觉：好像神经网络训练就是在一条或者一张起伏不平的曲面上寻找最低点。但真实情况完全不同。

如果一个模型有 $d$ 个参数，我们可以把所有参数写成一个向量：

$$
\theta = (\theta_1, \theta_2, \ldots, \theta_d)
$$

损失函数就是：

$$
L:\mathbb{R}^d \rightarrow \mathbb{R}
$$

也就是说，每一种参数配置都对应参数空间中的一个点，而这个点的高度就是当前的 loss。如果模型有十亿个参数，那么这个空间就是十亿维的。我们平时看到的二维 loss landscape，只能是这个巨大空间中的一个切片或者投影，而不是完整的损失函数。

因此，高维优化里最重要的一件事就是：**不要把二维地形图的直觉直接搬到高维空间。**

例如，在一维函数中，如果我们走到一个梯度为零的位置，通常只需要判断这里是山顶还是山谷。但在高维空间里，梯度为零只说明：

$$
\nabla L(\theta) = 0
$$

对，它只能告诉我们这里梯度为 0，并不能告诉我们这个点到底是什么。因此，我们还要看不同方向上的曲率。这些局部曲率可以通过 Hessian 矩阵描述：

$$
H = \nabla^2 L(\theta)
$$

如果把 Hessian 分解到不同方向上，我们可以得到一组特征值：

$$
\lambda_1,\lambda_2,\ldots,\lambda_d
$$

它们可以粗略理解成：从当前位置沿不同方向移动时，损失函数是向上弯还是向下弯。

| 临界点 | Hessian 的典型情况 | 直觉 |
|----|----|----|
| 局部极小值 | 所有方向都向上弯 | 往哪个方向走，loss 都会上升 |
| 局部极大值 | 所有方向都向下弯 | 往哪个方向走，loss 都会下降 |
| 鞍点 | 有些方向向上弯，有些方向向下弯 | 某些方向看起来像谷底，但另一些方向仍然可以下降 |

表 1.5.1 不同临界点的局部曲率

二维空间里的鞍点可以想象成马鞍：沿一个方向看，它像最低点；换一个方向看，它却像最高点。

<figure>
<img src="figures/ch1.5-saddle-point.svg" alt="图 1.5.1.2 二维空间中的鞍点 (Wikipedia contributors 2025)" width="65%" />
<figcaption aria-hidden="true">图 1.5.1.2 二维空间中的鞍点 <span class="citation" data-cites="enwiki:SaddlePoint">(Wikipedia contributors 2025)</span></figcaption>
</figure>

当参数维度非常高时，一个临界点需要在**所有方向**上都没有下降方向，才可能成为严格的局部极小值。只要还有某个方向的曲率为负，就仍然存在可以离开的方向。

这也是为什么高维非凸优化中，研究者很早就注意到：真正让优化变慢的地方不一定只是“坏的局部极小值”，大量鞍点和平坦区域同样重要。优化器可能来到一个梯度已经很小的位置，看起来几乎不再移动，但这并不代表它已经进入了一个真正的局部极小值。

不过，这里必须特别小心。我们不能因此得出：

> **参数维度很高，所以神经网络不会遇到局部极小值。**

这个结论太强了。真实神经网络的 loss landscape 具有非常复杂的结构，并不是一个随机生成的高维函数。高维几何只能帮助我们理解为什么“到处都是坏的孤立小坑”并不是唯一合理的图景，它并不能保证梯度下降一定找到全局最优解。

## 1.5.2 过参数化：我们可能根本不需要寻找唯一答案

还有一个非常反直觉的现象：神经网络参数很多，本来听起来应该让优化更加困难，但在现代深度学习中，**更多参数反而经常让训练变得更容易**。

为了理解这一点，我们先考虑一个非常简单的问题。假设我们只有两个参数 $\theta_1$ 和 $\theta_2$，但是训练数据要求它们满足三个彼此独立的条件。那么这三个条件可能根本无法同时满足：

$$
\begin{align}
f_1(\theta_1,\theta_2) &= 0,\\
f_2(\theta_1,\theta_2) &= 0,\\
f_3(\theta_1,\theta_2) &= 0,
\end{align}
$$

参数自由度太少，我们只能在几个条件之间做妥协。反过来，如果我们有很多参数，却只需要满足相对较少的约束，那么满足这些约束的参数配置可能不再只有一个点，而是一整片区域。

最简单的线性例子是：

$$
\theta_1 + \theta_2 = 1
$$

它不是只有一个解，而是有无穷多个：

$$
(0,1),\quad (0.2,0.8),\quad (2,-1),\quad \ldots
$$

所有解共同组成一条直线。

神经网络当然比这个例子复杂得多，但过参数化带来的直觉是类似的：当模型拥有大量自由度时，能够很好拟合训练数据的参数配置可能非常多。优化器不一定需要在整个参数空间里找到唯一的那个“正确答案”，而只需要进入某个足够好的解区域。

因此，神经网络训练不是在一个巨大山脉中寻找唯一最低的那个点，而更接近于：

> **参数空间中存在很多低 loss 的位置，甚至可能形成连续的低损失区域，我们只需要找到其中一个。**

这里还存在另一个原因，会让“唯一最优参数”这个概念变得不那么重要：**神经网络存在参数对称性。**

例如，一个隐藏层里有两个功能相同的神经元。如果我们交换这两个神经元的顺序，同时交换下一层与它们对应的连接，整个网络计算出的函数可以完全不变。也就是说：

$$
\theta_a \neq \theta_b
$$

但可能有：

$$
f(x;\theta_a) = f(x;\theta_b)
$$

所以，即使只考虑完全相同的函数，也可能存在很多不同的参数表示。训练神经网络的目标从来不是恢复某一组唯一的“真实参数”，而是找到一组能够实现我们需要的函数行为的参数。

这也是理解现代深度学习优化的一个重要转变：

> **参数多不只是负担，也会给优化过程提供更多自由度和更多可行路径。**

当然，过参数化也不是参数越多永远越好的定理。模型变大之后会带来更高的计算、显存和数据需求，而且不同网络结构的优化性质也不一样。这里我们只需要抓住一个核心直觉：在深度学习中，参数空间巨大并不意味着好解更加稀少；在很多情况下，恰恰相反，好解可能非常丰富。

## 1.5.3 为什么 SGD 不容易停在某个地方

前面讨论的是 loss landscape 的几何结构。接下来再看真正的训练算法。

如果我们使用完整数据集计算梯度，梯度下降得到的是：

$$
g = \nabla_\theta L(\theta)
$$

但实际训练神经网络时，我们几乎总是使用 mini-batch。假设当前 batch 是 $B$，那么我们计算的是：

$$
g_B = \nabla_\theta \frac{1}{|B|} \sum_{i\in B} L_i(\theta)
$$

不同 batch 包含的数据不同，因此通常有：

$$
g_B \neq \nabla_\theta L(\theta)
$$

也就是说，SGD 看到的梯度并不是完整 loss landscape 上完全精确的下降方向，而是一个带有随机性的估计。所以真实训练轨迹并不像一个小球沿着最陡方向平滑地下山，而更像“总体向下 + 持续随机抖动”。

这种随机性最初看起来像一个缺点，因为每一步都没有那么准确。但它有时反而会帮助优化。比如在一个非常平坦的区域附近，完整梯度可能已经非常小，而不同 mini-batch 产生的梯度仍然会有一些波动；在某些具有下降方向的鞍点附近，这些扰动也可能帮助参数离开当前区域。

不过，这里同样不能把结论说得太绝对。SGD 的噪声不保证模型一定能够逃离所有鞍点，更不保证它能够找到全局最优点。学习率、batch size、momentum、参数初始化方法以及网络结构都会改变整个优化过程。我们会在后面的优化算法章节里逐渐看到这些因素。真正值得记住的是：

> **神经网络训练不是一个确定的小球在固定曲面上机械滚动，而是一个高维、带随机性、不断受到数据和优化器共同影响的动态过程。**

这也是为什么两个完全相同的模型，即使使用相同的数据，只要随机初始化或者 mini-batch 顺序不同，最后得到的参数也可能完全不同。但它们最终实现的函数，却可能同样好。

## 1.5.4 数据虽然高维，却可能没有看起来那么复杂

到目前为止，我们一直讨论的是**参数空间**。但神经网络能够学习，还和另一个空间有关：**输入数据所在的空间**。

例如，一张 $224\times 224$ 的 RGB 图片包含：

$$
224\times 224\times 3 = 150528
$$

个数值。从形式上看，每张图片都是 $\mathbb{R}^{150528}$ 中的一个点。

这个空间有多大？如果每个像素都可以独立任意变化，那么几乎所有可能的点看起来都像随机噪声。真正的自然图像只占整个像素空间中极小的一部分。我们看到的人脸、动物、街景和文字并不是任意像素组合出来的。它们受到现实世界大量结构的约束：物体有连续的轮廓，相邻像素高度相关，光照变化通常是连续的，同一个物体旋转一点之后仍然是同一个物体。

这引出了机器学习中一个很有影响力的直觉：**流形假设（Manifold Hypothesis）**。它认为：

> **现实数据虽然表示在一个非常高维的空间中，但有效数据往往集中在某个维度低得多的结构附近。**

<figure>
<img src="figures/ch1.5-manifold-hypothesis.png" alt="图 1.5.4 流形假设示意图" width="70%" />
<figcaption aria-hidden="true">图 1.5.4 流形假设示意图</figcaption>
</figure>

如果这个直觉在某个任务上成立，那么模型其实不需要学会如何处理整个高维输入空间中的所有可能点。它主要需要在真实数据经常出现的那一小部分区域上学会正确的映射。这会让“学习一个高维函数”这件事没有表面上那么可怕。

比如图像分类模型不需要知道每一种 150528 维随机向量到底是什么，它只需要对自然图像附近的输入形成有用的决策边界。神经网络内部一层又一层的表示变换，也可以理解成不断把输入重新组织到更适合当前任务的表示空间里：

$$
x \rightarrow h_1 \rightarrow h_2 \rightarrow \cdots \rightarrow h_L
$$

对于分类任务，我们希望经过这些变换之后，原本纠缠在一起的数据变得更容易区分；对于生成任务，我们希望模型能够捕捉真实数据中那些稳定而有结构的变化规律。

不过，需要特别区分两个很容易混淆的概念。前面过参数化部分讨论的，是**参数空间中的解可能形成连续结构**；这里的流形假设讨论的，是**输入空间中的真实数据可能集中在低维结构附近**。它们都可能出现“流形”这个词，但说的是两件完全不同的事情。

而且，流形假设本身也不是一个已经对所有现实数据严格证明的定理。真实数据通常包含离散结构、噪声、多尺度变化和复杂拓扑，用一张光滑的低维流形描述所有数据往往过于理想化。把它理解成一种帮助我们思考“高维数据为什么仍然可能具有可学习结构”的假设，会更加准确。更重要的是，数据具有低维结构也不能直接证明神经网络优化一定容易。输入空间的结构回答的是“任务为什么可能存在规律”，而参数空间的几何和优化动力学回答的是“我们为什么可能找到实现这些规律的参数”。这两个问题相关，但不能混为一谈。

## 1.5.5 能把 loss 降下来，还不是最神奇的地方

到这里，我们已经可以对最开始的问题给出一个不那么神秘的回答。神经网络之所以能够被训练，至少有几个因素同时在起作用：

<figure>
<img src="figures/ch1.5-why-neural-networks-can-be-trained.svg" alt="图 1.5.5 为什么神经网络能够被训练" />
<figcaption aria-hidden="true">图 1.5.5 为什么神经网络能够被训练</figcaption>
</figure>

首先，神经网络由大量连续、几乎处处可导的运算组成，因此反向传播能够高效给出局部梯度。梯度虽然看不到整个参数空间，却能不断提供当前位置附近的改进方向。

其次，高维非凸空间并不只是布满一个个孤立的小坑。鞍点、平坦方向和复杂的低损失区域同样大量存在，因此“梯度下降一定会很快掉进坏的局部极小值”这个二维直觉并不可靠。

最后，现代神经网络通常高度过参数化。模型拥有大量自由度，能够完成训练任务的参数配置可能非常多，所以优化器往往不需要寻找某个唯一的精确答案。同时，mini-batch 带来的随机性让 SGD 的轨迹不会完全按照一个固定方向移动，而现实数据本身又具有大量结构，使得模型面对的并不是毫无规律的高维映射问题。

但如果你仔细看，会发现我们其实只回答了一个问题：

> **为什么训练损失能够被优化？**

深度学习真正更令人惊讶的问题是另一个：

> **为什么训练出来的神经网络还能泛化？**

一个参数量远大于训练样本数量的网络，往往拥有足够的能力把训练数据直接记住。如果我们的目标只是让训练 loss 下降，那么过参数化确实提供了很多帮助。但是为什么模型在把训练集拟合得非常好的同时，还经常能在从未见过的数据上做出正确预测？这件事不能仅靠“梯度下降找到了低 loss”来解释。

优化告诉我们如何从参数空间中找到一个解，而泛化关心的是：

> **为什么我们找到的这个解恰好对训练集之外的数据也有用。**

数据分布、网络结构的归纳偏置、优化器的隐式偏置、正则化、数据增强以及模型规模都会参与其中。这也是现代深度学习理论最有意思的地方之一。神经网络能够训练出来，并不代表我们已经完全理解了它为什么工作。

## 1.5.6 本章小结

这一节没有再介绍新的训练步骤，而是退后一步，讨论了一个更根本的问题：既然神经网络是一个巨大的非凸优化问题，为什么梯度方法仍然能够把它训练出来？

首先，我们不能用二维山谷图直接想象真实的 loss landscape。神经网络的参数空间可能有几百万甚至几十亿维，一个梯度接近零的位置不一定是局部极小值，也可能是鞍点或者非常平坦的区域。高维空间中存在大量不同方向，因此“优化器到处都会被坏的局部极小值困住”并不是一个足够准确的描述。

其次，现代神经网络通常是过参数化的。参数很多不仅意味着搜索空间更大，也意味着模型拥有更多自由度。能够很好拟合训练数据的解往往不是唯一的，因此优化器不一定要寻找某个孤立的最优点，而可能只需要进入众多低损失解中的一个。

SGD 本身的随机性也会改变优化轨迹。Mini-batch 梯度只是完整梯度的一个带噪估计，这些扰动有时可以帮助模型离开平坦区域或具有下降方向的鞍点。不过，它并不能保证训练一定成功，更不能保证找到全局最优解。

最后，我们还从输入空间看到了另一个重要直觉。现实数据虽然表示在高维空间中，但通常具有很强的内部结构。流形假设试图用“高维空间中的低维结构”描述这种现象。不过，数据流形和参数空间中的解结构是两个不同概念，而且数据具有低维结构本身也不能直接证明优化容易。

所以，神经网络能够被训练，并不是因为某一个神奇定理保证梯度下降一定成功，而是因为很多因素共同让这个问题比表面上更友好：网络可微、梯度有信息、高维空间存在丰富方向、过参数化提供大量可行解、SGD 带有随机性，而真实数据本身也并非毫无结构。

至此，我们已经完成了第一章最重要的任务：从“神经网络是一个可学习的函数”出发，一步步看到模型如何定义误差、如何计算梯度、如何更新参数，以及为什么这样一个看似不可思议的高维优化过程在实践中真的可以工作。

接下来，我们会进入 PyTorch，看看这些抽象概念在真正的深度学习框架中是如何被实现的。

Wikipedia contributors. 2025. *Saddle Point — Wikipedia, the Free Encyclopedia*. <https://en.wikipedia.org/w/index.php?title=Saddle_point&oldid=1285721453>.